# 11 — Ablation: LLM NER on German Text Directly

**Research question**: Is the pipeline advantage over LLMs fundamental to the architecture,
or is it an artefact of the translation bottleneck?

Tests Llama 3.3 70B on the **original German text** (`content` column) using the
same 183-article overlap set and same gold standard as notebooks 05 and 09.

| Condition | Input | Language | F1 (known) |
|-----------|-------|----------|------------|
| Flair pipeline | German | DE | 0.717 |
| Llama 70B (EN) | English (OPUS-MT) | EN | 0.340 |
| **Llama 70B (DE)** | **German (original)** | **DE** | **?** |

**What this isolates**: If Llama 70B (DE) significantly outperforms Llama 70B (EN),
the bottleneck is the translation step. If performance remains similar,
the bottleneck is the LLM architecture itself.

**Implication for cross-lingual accessibility**: If LLM-on-German closes the gap,
the trade-off becomes performance vs. language barrier — a researcher must
interact with German prompts and output. If it does not close the gap,
pipeline systems are simply better regardless of input language.


In [ ]:
!pip install -q groq pandas numpy matplotlib seaborn scipy scikit-learn
print('Done')

In [ ]:
import json
import pickle
import re
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from groq import Groq
from scipy import stats
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 130})
print('Imports OK')

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=False)
import yaml

PROJECT_ROOT = Path('/content/drive/MyDrive/thesis')
with open(PROJECT_ROOT / 'config.yaml') as f:
    config = yaml.safe_load(f)

SEED        = config.get('seed', 42)
DATA_PROC   = PROJECT_ROOT / 'Project' / 'Data' / 'Processed'
FIGURES_DIR = PROJECT_ROOT / 'Project' / 'Outputs' / 'Figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f'Config loaded | seed={SEED}')

In [ ]:
# Groq client
GROQ_TOKEN   = userdata.get('GROQ_TOKEN')
client       = Groq(api_key=GROQ_TOKEN)
ACTIVE_MODEL = 'llama-3.3-70b-versatile'

test = client.chat.completions.create(
    model=ACTIVE_MODEL,
    messages=[{'role': 'user', 'content': 'Antworte mit: OK'}],
    max_tokens=5,
)
print(f'Model: {ACTIVE_MODEL}')
print(f'Response: {test.choices[0].message.content.strip()}')

In [ ]:
# Load data
df = pd.read_pickle(DATA_PROC / 'ner_pipeline_results.pkl')
df = df[~df['exclude']].copy()

# Load LLM checkpoint from nb04 to get the 183-article overlap set
with open(DATA_PROC / 'ner_llm_checkpoint.pkl', 'rb') as f:
    nb04_results = pickle.load(f)

ok_ids = {aid for aid, r in nb04_results.items() if r['status'] == 'ok'}
df_val = df[df['article_id'].isin(ok_ids)].copy()

# Gold standard
agree_df = pd.read_csv(DATA_PROC / 'ner_agreement.csv')
gold_df  = agree_df[agree_df['agreement_level'] == 'full'].copy()
gold_df['entity_text_lower'] = gold_df['entity_text'].str.lower().str.strip()
gold_sets = (
    gold_df.groupby('article_id')
    .apply(lambda g: set(zip(g['entity_text_lower'], g['entity_label'])))
    .to_dict()
)
work_ids  = set(df_val['article_id'])
gold_sets = {k: v for k, v in gold_sets.items() if k in work_ids}

print(f'Overlap set   : {len(df_val)} articles')
print(f'Gold entities : {sum(len(v) for v in gold_sets.values())}')
print(f'German text available: {df_val["content"].notna().sum()}')

In [ ]:
# NER prompt in German
# Key difference from notebook 09: prompt is in German, input is German
# Output labels remain in English (PER/LOC/ORG/MISC) for comparability

NER_SYSTEM_PROMPT_DE = """Du bist ein präzises System zur Erkennung benannter Entitäten (Named Entity Recognition).
Extrahiere alle benannten Entitäten aus dem gegebenen Text.
Verwende NUR diese vier Labels:
  PER  – Personennamen
  LOC  – Orte, Regionen, Länder, Städte
  ORG  – Organisationen, Unternehmen, Institutionen, Behörden
  MISC – Andere benannte Entitäten (Produkte, Ereignisse, Gesetze, Standards, Auszeichnungen)

Regeln:
- Vollständige Entitätsspannen angeben (z.B. "Bundesministerium für Wirtschaft" nicht nur "Ministerium")
- KEINE gewöhnlichen Nomen, Adjektive oder generische Begriffe
- Dieselbe Entität NICHT zweimal angeben, außer sie erscheint mit unterschiedlichem Text
- Antworte NUR mit einem JSON-Array. Keine Erklärung, kein Markdown.

Format:
[{"text": "<Entitätstext>", "label": "<PER|LOC|ORG|MISC>"}, ...]

Wenn keine Entitäten gefunden werden: []"""

VALID_LABELS = {'PER', 'LOC', 'ORG', 'MISC'}


def parse_response(response_text):
    text = response_text.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text).strip()
    match = re.search(r'(\[.*?\])', text, re.DOTALL)
    if match:
        text = match.group(1)
    try:
        entities = json.loads(text)
    except json.JSONDecodeError:
        return []
    if not isinstance(entities, list):
        return []
    clean = []
    for e in entities:
        if not isinstance(e, dict):
            continue
        t = str(e.get('text', '')).strip()
        l = str(e.get('label', '')).strip().upper()
        if t and l in VALID_LABELS:
            clean.append({'text': t, 'label': l})
    return clean


def add_offsets(entities, source_text):
    result    = []
    src_lower = source_text.lower()
    for ent in entities:
        term = ent['text'].lower()
        idx  = src_lower.find(term)
        result.append({
            'text' : ent['text'],
            'label': ent['label'],
            'start': idx,
            'end'  : idx + len(ent['text']) if idx != -1 else -1,
        })
    return result


print('German NER prompt defined')
print(f'Prompt length: {len(NER_SYSTEM_PROMPT_DE)} chars')

In [ ]:
# Inference on German text
CHECKPOINT_DE = DATA_PROC / 'ner_llama70b_de_checkpoint.pkl'


def load_ckpt(path):
    if path.exists():
        with open(path, 'rb') as f:
            d = pickle.load(f)
        print(f'Checkpoint loaded: {len(d)} articles')
        return d
    return {}


def save_ckpt(path, d):
    with open(path, 'wb') as f:
        pickle.dump(d, f)


results_de    = load_ckpt(CHECKPOINT_DE)
todo          = df_val[~df_val['article_id'].isin(results_de.keys())]

print(f'Total    : {len(df_val)}')
print(f'Done     : {len(results_de)}')
print(f'Remaining: {len(todo)}')
print(f'Model    : {ACTIVE_MODEL} (German input)')
print(f'Est. time: ~{len(todo) * 2 / 60:.1f} min')

status_counts = {'ok': 0, 'api_error': 0}

for i, (_, row) in enumerate(tqdm(todo.iterrows(), total=len(todo),
                                   desc='NER Llama 70B (DE)')):
    aid        = row['article_id']
    content_de = str(row.get('content', ''))

    # Truncate to ~3000 words to stay within context
    words = content_de.split()
    if len(words) > 3000:
        content_de = ' '.join(words[:3000])

    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=ACTIVE_MODEL,
                messages=[
                    {'role': 'system', 'content': NER_SYSTEM_PROMPT_DE},
                    {'role': 'user',
                     'content': f'Extrahiere benannte Entitäten:\n\n{content_de}'},
                ],
                max_tokens=1024,
                temperature=0.0,
            )
            raw      = response.choices[0].message.content
            entities = parse_response(raw)
            entities = add_offsets(entities, content_de)
            results_de[aid] = {
                'entities'           : entities,
                'entity_count'       : len(entities),
                'status'             : 'ok',
                'model'              : ACTIVE_MODEL,
                'input_language'     : 'de',
                'translation_quality': row.get('translation_quality', 'unknown'),
            }
            status_counts['ok'] += 1
            break
        except Exception as e:
            if attempt < 2:
                time.sleep(10 * (attempt + 1))
            else:
                results_de[aid] = {
                    'entities': [], 'entity_count': 0,
                    'status': 'api_error', 'input_language': 'de',
                    'translation_quality': row.get('translation_quality', 'unknown'),
                }
                status_counts['api_error'] += 1
                tqdm.write(f'  Error on {aid}: {e}')

    if (i + 1) % 10 == 0:
        save_ckpt(CHECKPOINT_DE, results_de)
        tqdm.write(f'  Checkpoint saved ({len(results_de)} done)')

    time.sleep(2.0)

save_ckpt(CHECKPOINT_DE, results_de)
print(f'\nDone: {status_counts}')

In [ ]:
# Evaluate Llama 70B (DE) — same metrics as all other notebooks

def normalise(ents):
    if not isinstance(ents, list):
        return set()
    return {
        (str(e.get('text', '')).lower().strip(),
         str(e.get('label', '')).upper().strip())
        for e in ents
        if isinstance(e, dict)
        and str(e.get('text', '')).strip()
        and str(e.get('label', '')).upper().strip() in VALID_LABELS
    }


def prf(predicted, gold):
    if not gold:
        return 0.0, 0.0, 0.0
    tp = len(predicted & gold)
    fp = len(predicted - gold)
    fn = len(gold - predicted)
    p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1


rows_de = []
for _, row in df_val.iterrows():
    aid   = row['article_id']
    gold  = gold_sets.get(aid, set())
    r_de  = results_de.get(aid, {})
    if r_de.get('status') != 'ok':
        continue
    pred = normalise(r_de.get('entities', []))
    p, r, f1 = prf(pred, gold)
    rows_de.append({
        'article_id'         : aid,
        'p'                  : p,
        'r'                  : r,
        'f1'                 : f1,
        'pred_count'         : len(pred),
        'gold_count'         : len(gold),
        'source'             : row['source'],
        'year_bin'           : row['year_bin'],
        'word_count'         : row['word_count'],
        'compound_count'     : row['compound_count'],
        'translation_quality': row.get('translation_quality', 'unknown'),
    })

res_de = pd.DataFrame(rows_de)

print(f'Articles evaluated : {len(res_de)}')
print(f'Mean Precision     : {res_de["p"].mean():.3f}')
print(f'Mean Recall        : {res_de["r"].mean():.3f}')
print(f'Mean F1            : {res_de["f1"].mean():.3f}')
print(f'F1 std             : {res_de["f1"].std():.3f}')

In [ ]:
# Bootstrap CI
np.random.seed(SEED)
f1_vals    = res_de['f1'].values
boot_means = [
    np.mean(np.random.choice(f1_vals, size=len(f1_vals), replace=True))
    for _ in range(2000)
]
ci_lower = np.percentile(boot_means, 2.5)
ci_upper = np.percentile(boot_means, 97.5)

f1_de_mean = res_de['f1'].mean()
print(f'Llama 70B (DE) F1 = {f1_de_mean:.3f} (95% CI: {ci_lower:.3f}–{ci_upper:.3f})')

# Per-label F1 for Llama 70B (DE)
print('\nPer-label F1 (Llama 70B DE):')
for label in ['PER', 'LOC', 'ORG', 'MISC']:
    tp = fp = fn = 0
    for _, row in df_val.iterrows():
        aid  = row['article_id']
        r_de = results_de.get(aid, {})
        if r_de.get('status') != 'ok':
            continue
        gold = {e for e in gold_sets.get(aid, set()) if e[1] == label}
        pred = {e for e in normalise(r_de.get('entities', [])) if e[1] == label}
        tp  += len(pred & gold)
        fp  += len(pred - gold)
        fn  += len(gold - pred)
    p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    print(f'  {label:4s}: P={p:.3f}  R={r:.3f}  F1={f1:.3f}')

In [ ]:
# Load comparison values from previous notebooks
with open(DATA_PROC / 'ner_comparison_summary.json') as f:
    nb05 = json.load(f)
with open(DATA_PROC / 'nb09_llama70b_summary.json') as f:
    nb09 = json.load(f)

spacy_f1  = nb05['macro_f1']['spaCy (DE)']
stanza_f1 = nb05['macro_f1']['Stanza (DE)']
flair_f1  = nb05['macro_f1']['Flair (DE)']
qwen_f1   = nb05['macro_f1']['Qwen (EN)']
llama_en  = nb09['ner_macro_f1']
llama_de  = float(res_de['f1'].mean())

print('── Full NER Comparison ──')
print(f'spaCy (DE)          : {spacy_f1:.3f}  [pipeline]')
print(f'Stanza (DE)         : {stanza_f1:.3f}  [pipeline]')
print(f'Flair (DE)          : {flair_f1:.3f}  [pipeline]')
print(f'Qwen 7B (EN)        : {qwen_f1:.3f}  [LLM, small, EN input]')
print(f'Llama 70B (EN)      : {llama_en:.3f}  [LLM, large, EN input]')
print(f'Llama 70B (DE)      : {llama_de:.3f}  [LLM, large, DE input]')
print()
print(f'Translation effect  : {llama_de - llama_en:+.3f} F1  (DE vs EN input, same model)')
print(f'Remaining gap       : {flair_f1 - llama_de:+.3f} F1  (Flair vs Llama 70B DE)')
print()
if llama_de > llama_en:
    print('FINDING: German input improves LLM performance — translation is a bottleneck.')
    if llama_de >= flair_f1:
        print('FINDING: Llama 70B (DE) matches or beats Flair — gap closes with native input.')
    else:
        print('FINDING: Pipeline advantage persists even with German input — gap is partly architectural.')
else:
    print('FINDING: German input does not improve LLM performance — gap is architectural, not translation.')

In [ ]:
# Statistical test: Llama 70B (DE) vs Llama 70B (EN)
with open(DATA_PROC / 'ner_llama70b_checkpoint.pkl', 'rb') as f:
    llama_en_results = pickle.load(f)

# Build per-article F1 for Llama EN on same articles
rows_en = []
for _, row in df_val.iterrows():
    aid  = row['article_id']
    gold = gold_sets.get(aid, set())
    r_en = llama_en_results.get(aid, {})
    if r_en.get('status') != 'ok':
        continue
    pred = normalise(r_en.get('entities', []))
    _, _, f1 = prf(pred, gold)
    rows_en.append({'article_id': aid, 'f1_en': f1})

res_en = pd.DataFrame(rows_en).set_index('article_id')
res_de_idx = res_de.set_index('article_id')

# Align on common articles
common = res_en.index.intersection(res_de_idx.index)
f1_en_aligned = res_en.loc[common, 'f1_en']
f1_de_aligned = res_de_idx.loc[common, 'f1']

stat, p = stats.wilcoxon(f1_de_aligned, f1_en_aligned)
print(f'Wilcoxon test: Llama 70B (DE) vs Llama 70B (EN)')
print(f'  N paired articles : {len(common)}')
print(f'  Mean F1 (DE)      : {f1_de_aligned.mean():.3f}')
print(f'  Mean F1 (EN)      : {f1_en_aligned.mean():.3f}')
print(f'  Difference        : {f1_de_aligned.mean() - f1_en_aligned.mean():+.3f}')
print(f'  p-value           : {p:.4f} {"* significant" if p < 0.05 else "not significant"}')

# Also test Llama 70B (DE) vs Flair
ner_compare = pd.read_csv(DATA_PROC / 'ner_comparison_results.csv')
flair_f1_per = ner_compare[ner_compare['system'] == 'Flair (DE)'].set_index('article_id')['f1']
common_flair = flair_f1_per.index.intersection(res_de_idx.index)
stat2, p2 = stats.wilcoxon(
    flair_f1_per.loc[common_flair],
    res_de_idx.loc[common_flair, 'f1']
)
print(f'\nWilcoxon test: Flair (DE) vs Llama 70B (DE)')
print(f'  Difference: {flair_f1_per.loc[common_flair].mean() - res_de_idx.loc[common_flair, "f1"].mean():+.3f}')
print(f'  p-value   : {p2:.4f} {"* significant" if p2 < 0.05 else "not significant"}')

In [ ]:
# Figure 1: The ablation result — all conditions
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Ablation: Does Input Language Explain the Pipeline Advantage?',
             fontsize=13, fontweight='bold')

# Left: all 6 conditions
ax = axes[0]
conditions = [
    ('spaCy\n(DE)',       spacy_f1,  '#2196F3', 'solid'),
    ('Stanza\n(DE)',      stanza_f1, '#4CAF50', 'solid'),
    ('Flair\n(DE)',       flair_f1,  '#FF9800', 'solid'),
    ('Qwen 7B\n(EN)',     qwen_f1,   '#9C27B0', 'solid'),
    ('Llama 70B\n(EN)',   llama_en,  '#E91E63', 'solid'),
    ('Llama 70B\n(DE)',   llama_de,  '#E91E63', 'solid'),
]

names   = [c[0] for c in conditions]
f1s     = [c[1] for c in conditions]
colors  = [c[2] for c in conditions]

bars = ax.bar(names, f1s, color=colors, alpha=0.85, edgecolor='white')
# Make Llama DE bar hatched to distinguish from EN
bars[-1].set_hatch('///')
bars[-1].set_alpha(0.6)

for bar, f1 in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{f1:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.axvline(2.5, color='gray', linestyle='--', alpha=0.4, label='Pipeline | LLM')
ax.set_ylabel('Macro-averaged F1')
ax.set_title('All Conditions')
ax.set_ylim(0, 0.92)
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=8)

# Add annotation for Llama DE hatch
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E91E63', alpha=0.85, label='Llama 70B (EN input)'),
    Patch(facecolor='#E91E63', alpha=0.6,  hatch='///', label='Llama 70B (DE input)'),
]
ax.legend(handles=legend_elements, fontsize=8, loc='upper left')

# Right: focused comparison — translation effect
ax2 = axes[1]
focus_names = ['Llama 70B\n(EN input)', 'Llama 70B\n(DE input)', 'Flair\n(DE pipeline)']
focus_f1s   = [llama_en, llama_de, flair_f1]
focus_cols  = ['#E91E63', '#E91E63', '#FF9800']
focus_alpha = [0.85, 0.6, 0.85]
focus_hatch = ['', '///', '']

for i, (name, f1, col, alpha, hatch) in enumerate(
        zip(focus_names, focus_f1s, focus_cols, focus_alpha, focus_hatch)):
    bar = ax2.bar(i, f1, color=col, alpha=alpha, edgecolor='white',
                  hatch=hatch, width=0.5)
    ax2.text(i, f1 + 0.01, f'{f1:.3f}',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

# Annotate translation effect
trans_diff = llama_de - llama_en
ax2.annotate('', xy=(1, llama_de), xytext=(0, llama_en),
             arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax2.text(0.5, (llama_en + llama_de)/2 + 0.01,
         f'{trans_diff:+.3f}\n(DE input\neffect)',
         ha='center', fontsize=9)

# Annotate remaining pipeline gap
remain_diff = flair_f1 - llama_de
ax2.annotate('', xy=(2, flair_f1), xytext=(1, llama_de),
             arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax2.text(1.5, (llama_de + flair_f1)/2 + 0.01,
         f'{remain_diff:+.3f}\n(remaining\npipeline gap)',
         ha='center', fontsize=9)

ax2.set_xticks(range(3))
ax2.set_xticklabels(focus_names, fontsize=9)
ax2.set_ylabel('Macro-averaged F1')
ax2.set_title('Translation Effect Decomposition')
ax2.set_ylim(0, 0.92)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'ablation_language_input.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved')

In [ ]:
# Figure 2: Per-label comparison — EN vs DE input for Llama 70B
# Load per-label F1 for Llama 70B EN from error analysis
with open(DATA_PROC / 'error_analysis_summary.json') as f:
    error_summary = json.load(f)

llama_en_label = error_summary['per_label_f1']['Llama 70B']

# Compute per-label F1 for Llama 70B DE
llama_de_label = {}
for label in ['PER', 'LOC', 'ORG', 'MISC']:
    tp = fp = fn = 0
    for _, row in df_val.iterrows():
        aid  = row['article_id']
        r_de = results_de.get(aid, {})
        if r_de.get('status') != 'ok':
            continue
        gold = {e for e in gold_sets.get(aid, set()) if e[1] == label}
        pred = {e for e in normalise(r_de.get('entities', [])) if e[1] == label}
        tp  += len(pred & gold)
        fp  += len(pred - gold)
        fn  += len(gold - pred)
    p_v  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r_v  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_v = 2 * p_v * r_v / (p_v + r_v) if (p_v + r_v) > 0 else 0.0
    llama_de_label[label] = round(f1_v, 3)

flair_label = error_summary['per_label_f1']['Flair (DE)']

labels_order = ['PER', 'LOC', 'ORG', 'MISC']
x     = np.arange(len(labels_order))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, [flair_label[l]    for l in labels_order],
       width, label='Flair (DE pipeline)', color='#FF9800', alpha=0.85)
ax.bar(x,         [llama_en_label[l] for l in labels_order],
       width, label='Llama 70B (EN input)', color='#E91E63', alpha=0.85)
ax.bar(x + width, [llama_de_label[l] for l in labels_order],
       width, label='Llama 70B (DE input)', color='#E91E63', alpha=0.5, hatch='///')

ax.set_xticks(x)
ax.set_xticklabels(labels_order, fontsize=12)
ax.set_ylabel('F1 Score')
ax.set_title('Per-label F1: Does German Input Help LLMs?',
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

# Annotate LOC improvement specifically
loc_idx = labels_order.index('LOC')
ax.annotate(
    f"+{llama_de_label['LOC'] - llama_en_label['LOC']:.3f}",
    xy=(loc_idx + width, llama_de_label['LOC']),
    xytext=(loc_idx + width + 0.15, llama_de_label['LOC'] + 0.05),
    fontsize=9, color='black',
    arrowprops=dict(arrowstyle='->', color='black', lw=1)
)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'ablation_per_label.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved')
print('\nPer-label comparison:')
for l in labels_order:
    diff = llama_de_label[l] - llama_en_label[l]
    print(f'  {l}: Flair={flair_label[l]:.3f}  '
          f'LlamaEN={llama_en_label[l]:.3f}  '
          f'LlamaDE={llama_de_label[l]:.3f}  '
          f'(DE-EN: {diff:+.3f})')

In [ ]:
# Summary interpretation
print('=' * 65)
print('ABLATION STUDY FINDINGS')
print('=' * 65)

trans_effect = llama_de - llama_en
remain_gap   = flair_f1 - llama_de

print(f"""
DECOMPOSING THE PIPELINE ADVANTAGE:

Total pipeline advantage (Flair vs Qwen 7B EN): {flair_f1 - qwen_f1:+.3f} F1

Explained by model scale (7B → 70B)          : {llama_en - qwen_f1:+.3f} F1
Explained by input language (EN → DE)         : {trans_effect:+.3f} F1
Remaining unexplained gap (Flair vs Llama DE) : {remain_gap:+.3f} F1

INTERPRETATION:
""")

if trans_effect > 0.05 and p < 0.05:
    print(f"""  Translation IS a significant bottleneck (p={p:.4f}).
  Switching from English to German input improves Llama 70B F1
  by {trans_effect:+.3f} points — a meaningful gain.

  However, even with German input, Llama 70B {'matches' if remain_gap < 0.02
  else f'still falls {remain_gap:.3f} F1 points below'} Flair.

  CONCLUSION: The pipeline advantage has TWO sources:
  (1) Translation loss — partially recoverable by using German input
  (2) Architectural difference — task-specific models trained on German
      NER data outperform general-purpose LLMs on this domain
      regardless of input language."""
)
elif trans_effect > 0.05 and p >= 0.05:
    print(f"""  Translation appears to help (+{trans_effect:.3f}) but the
  difference is not statistically significant (p={p:.4f}).
  The pipeline advantage appears architectural rather than
  purely a translation artefact."""
)
else:
    print(f"""  German input does NOT significantly improve LLM performance
  (Δ={trans_effect:+.3f}, p={p:.4f}).
  The pipeline advantage is architectural — task-specific models
  fundamentally outperform general LLMs on German NER regardless
  of whether the LLM receives German or English input."""
)

In [ ]:
# Save ablation summary
ablation_summary = {
    'model'                    : ACTIVE_MODEL,
    'n_articles'               : int(len(res_de)),
    'llama_70b_de': {
        'macro_f1'       : round(float(res_de['f1'].mean()), 4),
        'macro_precision': round(float(res_de['p'].mean()), 4),
        'macro_recall'   : round(float(res_de['r'].mean()), 4),
        'ci_lower'       : round(float(ci_lower), 4),
        'ci_upper'       : round(float(ci_upper), 4),
        'per_label_f1'   : llama_de_label,
    },
    'comparison': {
        'flair_de'    : flair_f1,
        'llama_70b_en': llama_en,
        'llama_70b_de': round(float(res_de['f1'].mean()), 4),
    },
    'translation_effect': {
        'f1_gain'       : round(float(llama_de - llama_en), 4),
        'wilcoxon_p'    : round(float(p), 4),
        'significant'   : bool(p < 0.05),
    },
    'remaining_pipeline_gap': round(float(flair_f1 - llama_de), 4),
    'flair_vs_llama_de_p'   : round(float(p2), 4),
}

with open(DATA_PROC / 'ablation_summary.json', 'w') as f:
    json.dump(ablation_summary, f, indent=2)

print('Saved: ablation_summary.json')
print(json.dumps(ablation_summary, indent=2))

## Notebook summary

| Item | Detail |
|------|--------|
| Model | Llama 3.3 70B (Groq) |
| Input | German original text (`content` column) |
| Prompt | German-language NER prompt |
| Output labels | PER / LOC / ORG / MISC (English, for comparability) |
| Gold standard | Full cross-model agreement (same as all other notebooks) |
| Saved | `ner_llama70b_de_checkpoint.pkl`, `ablation_summary.json` |
| Figures | `ablation_language_input.png`, `ablation_per_label.png` |

### What this answers

The pipeline advantage decomposes into:
1. **Scale effect** (7B → 70B): quantified in notebook 09
2. **Translation effect** (EN → DE input): quantified here
3. **Architectural gap** (task-specific vs general LLM): the remainder

This is the cleanest possible answer to the research question.

**Next**: Update notebook 07 with ablation results for final synthesis.
